# Assignment Case C — Customer RFM Analysis
## Working with PySpark DataFrames for Analytics
**Day 42 | Batch 13 | 15 Mei 2026**

---

| Field | Isi |
|-------|-----|
| **Nama** | *(Fakhri Maulana Assiddiq)* |
| **NIM** | *(DE Batch 13)* |
| **Case** | Case C — Customer RFM Segmentation |
| **Source** | Hive: `adventureworks.*` |
| **Target** | Hive: `adventureworks_curated.fact_customer_rfm` |

---

## 🎯 Business Question

1. **Siapa customer paling berharga** (Champion & Loyal)?

Berdasarkan data, customer yang masuk segment champion berjumlah 3719 orang dan customer loyal 2579 orang, dan untuk rata rata spending Champion ada di 24K dollar per customer, sedangkan Loyal 10K dollar per customer. Top spender customer yang paling berharga adalah Roger Harui dengan total belanja Sekitar \$989K dan total order 12 kali.

2. **Customer mana yang perlu di-winback** (At Risk & Lost)?

Berdasarkan data, segment lost berjumlah 3539 customer. tapi terdapat customer di Southwest berjumlah 1075 orang dan Northwest 788 orang yang menurut saya masih layak di winback kembali karena total spending mereka yang cukup tinggi di angka 800+.

3. **Distribusi segmen** — berapa % customer di tiap segmen?

Berdasarkan data, mayoritas pelanggan kita justru berada di segmen potential Loyal, yaitu sebanyak 5840 orang atau 30,55% dari total customer. Sisanya terbagi ke segmen Champion 19,45%, Lost 18,51%, dan sisanya 18,00%, dan porsi terkecil ada di segmen Loyal 13,49%.

4. **Rata-rata revenue** per segmen RFM?

rata rata revenue per pelanggan didominasi oleh segmen Champion 24099 dollar dan Loyal 10459 dollar. 

## 📐 RFM Framework

```
R — Recency   : Berapa hari sejak order terakhir? (lebih kecil = lebih baik)
F — Frequency : Berapa kali order? (lebih besar = lebih baik)  
M — Monetary  : Berapa total spending? (lebih besar = lebih baik)
```

## 📊 Scoring 1–4 (Quartile-based)

| Dimensi | Score 4 (Terbaik) | Score 1 (Terburuk) |
|---------|-------------------|--------------------|
| **R** | Recency terkecil (baru beli) | Recency terbesar (lama tidak beli) |
| **F** | Frequency terbesar | Frequency terkecil |
| **M** | Monetary terbesar | Monetary terkecil |

> ⚠️ Recency di-score **terbalik**: recency kecil → score R tinggi

## 🏷️ Segmentasi

| Segment | Kondisi |
|---------|--------|
| **Champion** | R ≥ 3 AND F ≥ 3 AND M ≥ 3 |
| **Loyal** | F ≥ 3 AND M ≥ 3 (bukan Champion) |
| **Potential Loyal** | R ≥ 3 AND F >= 2 |
| **At Risk** | R <= 2 AND F >= 3 AND M >= 3 |
| **Lost** | R = 1 AND F <= 2 |
| **New Customer** | R = 4 AND F = 1 |
| **Others** | Semua yang tidak masuk kategori di atas |

## 📋 Kolom Target: `fact_customer_rfm`

| Kolom | Tipe | Keterangan |
|-------|------|------------|
| `customer_id` | INT | ID customer |
| `customer_name` | STRING | Nama customer |
| `territory_name` | STRING | Wilayah customer |
| `last_order_date` | DATE | Tanggal order terakhir |
| `recency_days` | INT | Hari sejak order terakhir |
| `frequency` | INT | Jumlah order |
| `monetary` | DECIMAL | Total spending |
| `r_score` | INT | Recency score (1-4) |
| `f_score` | INT | Frequency score (1-4) |
| `m_score` | INT | Monetary score (1-4) |
| `rfm_score` | STRING | Gabungan R+F+M score, contoh: '344' |
| `rfm_segment` | STRING | Champion/Loyal/At Risk/dll |
| `load_timestamp` | TIMESTAMP | Waktu load |


## 0. Persiapan: Buat Hive External Tables untuk RFM

Sebelum mulai analisis, kamu perlu menyiapkan tabel sumber sebagai
**Hive External Tables**. Pola tutorialnya sama dengan:
- `02_adventureworks_to_hdfs.ipynb` → extract PostgreSQL ke HDFS Parquet (dimensi + fakta dengan partisi)
- `03_hdfs_to_hive.ipynb` → buat External Tables di atas Parquet

**Tabel yang akan dibuat di Hive (`adventureworks.*`):**
| Hive Table | Sumber PostgreSQL | Catatan |
|---|---|---|
| `fact_sales_orders` | `Sales.SalesOrderHeader` | partisi `order_year`, `order_month` |
| `dim_customer` | `Sales.Customer` | |
| `dim_person` | `Person.Person` | |
| `dim_territory` | `Sales.SalesTerritory` | |

> **Lewati bagian ini** jika tabel sudah ada di Hive.

### 0.1 Setup SparkSession JDBC (untuk Extract ke HDFS)

Mirip dengan `02_adventureworks_to_hdfs.ipynb`. SparkSession ini
**tanpa Hive support** — hanya untuk extract data dari PostgreSQL ke HDFS sebagai Parquet.

In [1]:
# TODO: setup environment & import
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--jars /home/jovyan/work/jars/postgresql-42.7.3.jar pyspark-shell'  # TODO: '--jars /home/jovyan/work/jars/postgresql-42.7.3.jar pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F_jdbc

# TODO: buat spark_jdbc tanpa Hive support
spark_jdbc = SparkSession.builder \
    .master('local[*]') \
    .appName('RFM-to-HDFS') \
    .config('spark.sql.parquet.compression.codec', 'snappy') \
    .getOrCreate()

# TODO: konfigurasi JDBC PostgreSQL AdventureWorks
JDBC_URL   = 'jdbc:postgresql://adventureworks-postgres:5432/postgres'  # TODO: 'jdbc:postgresql://adventureworks-postgres:5432/postgres'
JDBC_PROPS = {
    "user": "postgres",
    "password": "My_password1",
    "driver": "org.postgresql.Driver"
}  # TODO: dict berisi 'user', 'password', 'driver'
HDFS_BASE  = 'hdfs://namenode:9000/datalake/raw'  # TODO: 'hdfs://namenode:9000/datalake/raw'

def read_pg(schema, table):
    return spark_jdbc.read.jdbc(
        url=JDBC_URL,
        table=f'"{schema}"."{table}"',
        properties=JDBC_PROPS
    )

print(f'spark_jdbc {spark_jdbc.version} ready')


spark_jdbc 3.5.0 ready


### 0.2 Extract Tabel Dimensi ke HDFS

Tabel dimensi (Customer, Person, SalesTerritory) tanpa partisi.

In [2]:
# TODO: lengkapi list tabel dimensi yang dibutuhkan RFM
dim_tables = [
    ('Sales',  'Customer'),  # TODO: ('Sales',  'Customer')
    ('Person', 'Person'),  # TODO: ('Person', 'Person')
    ('Sales',  'SalesTerritory'),  # TODO: ('Sales',  'SalesTerritory')
]

for schema, table in dim_tables:
    df = read_pg(schema, table)  # TODO: read_pg(schema, table)
    df = df.toDF(*[c.lower() for c in df.columns])
    hdfs_path = f'{HDFS_BASE}/{schema}/{table}'  # TODO: f'{HDFS_BASE}/{schema}/{table}'
    df.write.mode('overwrite').parquet(hdfs_path)
    print(f'  {schema}.{table:20s} -> {df.count():6,} rows -> HDFS')

print('Tabel dimensi selesai disimpan!')


  Sales.Customer             -> 19,820 rows -> HDFS
  Person.Person               -> 19,972 rows -> HDFS
  Sales.SalesTerritory       ->     10 rows -> HDFS
Tabel dimensi selesai disimpan!


### 0.3 Extract Tabel Fakta dengan Partisi

`SalesOrderHeader` adalah fakta utama RFM. Partisi by `order_year` dan `order_month`.

In [3]:
from pyspark.sql.types import DecimalType

# TODO: Baca tabel SalesOrderHeader
df_oh = read_pg('Sales', 'SalesOrderHeader')

# Paksa semua nama kolom menjadi huruf kecil
df_oh = df_oh.toDF(*[c.lower() for c in df_oh.columns])

df_oh = df_oh.withColumn('totaldue', 
    F_jdbc.col('subtotal') + F_jdbc.col('taxamt') + F_jdbc.col('freight'))

# KUNCI UTAMA: Paksa kolom uang menjadi persis DECIMAL(19,4) agar 100% cocok dengan Hive
uang_cols = ['subtotal', 'taxamt', 'freight', 'totaldue']
for c in uang_cols:
    df_oh = df_oh.withColumn(c, F_jdbc.col(c).cast(DecimalType(19,4)))

# TODO: Tambahkan kolom order_year dan order_month untuk partisi
df_oh_part = df_oh \
    .withColumn('order_year',  F_jdbc.year('orderdate')) \
    .withColumn('order_month', F_jdbc.month('orderdate'))

orders_path = f'{HDFS_BASE}/Sales/SalesOrderHeader'

# TODO: Simpan ke HDFS dengan PARTITIONED BY order_year, order_month
df_oh_part.write.mode('overwrite') \
    .partitionBy('order_year', 'order_month') \
    .parquet(orders_path)

print(f'  SalesOrderHeader -> {df_oh.count():,} rows -> HDFS (partisi order_year/order_month)')

  SalesOrderHeader -> 31,465 rows -> HDFS (partisi order_year/order_month)


### 0.4 Setup SparkSession dengan Hive Support

Mirip dengan `03_hdfs_to_hive.ipynb`. Tutup `spark_jdbc` dulu, lalu buat
SparkSession baru **dengan** Hive support untuk membuat External Tables.

In [4]:
# TODO: stop spark_jdbc agar tidak konflik resource
spark_jdbc.stop()

from pyspark.sql import SparkSession

# TODO: buat spark_hive dengan Hive support
spark_hive = SparkSession.builder \
    .master('local[*]') \
    .appName('RFM-HiveSetup') \
    .config('spark.jars', '/home/jovyan/work/jars/postgresql-42.7.3.jar') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .enableHiveSupport() \
    .getOrCreate()

HDFS_BASE = 'hdfs://namenode:9000/datalake/raw'

# TODO: buat database adventureworks jika belum ada
spark_hive.sql('CREATE DATABASE IF NOT EXISTS adventureworks')  # TODO: CREATE DATABASE IF NOT EXISTS adventureworks
spark_hive.sql('USE adventureworks')
print(f'spark_hive {spark_hive.version} ready')
spark_hive.sql('SHOW DATABASES').show()


spark_hive 3.5.0 ready
+--------------------+
|           namespace|
+--------------------+
|      adventureworks|
|adventureworks_cu...|
|             default|
+--------------------+



### 0.5 Buat Hive External Tables — Tabel Dimensi

Tabel dimensi tanpa partisi → schema otomatis di-infer dari Parquet.

In [5]:
# Helper function — sama persis dengan pola di 03_hdfs_to_hive.ipynb
def create_external_table(table_name, hdfs_path):
    spark_hive.sql(f'DROP TABLE IF EXISTS adventureworks.{table_name}')
    spark_hive.sql(f"""
        CREATE EXTERNAL TABLE adventureworks.{table_name}
        USING PARQUET
        LOCATION '{hdfs_path}'
    """)
    count = spark_hive.sql(f'SELECT COUNT(*) FROM adventureworks.{table_name}').collect()[0][0]
    print(f'  {table_name:35s} -> {count:6,} rows')

# TODO: lengkapi mapping nama tabel Hive -> path HDFS
hive_dim_tables = [
    ('dim_customer',  f'{HDFS_BASE}/Sales/Customer'),  # TODO: ('dim_customer',  f'{HDFS_BASE}/Sales/Customer')
    ('dim_person',    f'{HDFS_BASE}/Person/Person'),  # TODO: ('dim_person',    f'{HDFS_BASE}/Person/Person')
    ('dim_territory', f'{HDFS_BASE}/Sales/SalesTerritory'),  # TODO: ('dim_territory', f'{HDFS_BASE}/Sales/SalesTerritory')
]

for tname, path in hive_dim_tables:
    create_external_table(tname, path)  # TODO

print('Tabel dimensi Hive selesai!')


  dim_customer                        -> 19,820 rows
  dim_person                          -> 19,972 rows
  dim_territory                       ->     10 rows
Tabel dimensi Hive selesai!


### 0.6 Buat Hive External Table — Tabel Fakta dengan Partisi

Buat `fact_sales_orders` dengan skema eksplisit + `PARTITIONED BY (order_year, order_month)`,
lalu jalankan `MSCK REPAIR TABLE`. Pola sama dengan `03_hdfs_to_hive.ipynb`.

In [6]:
orders_path = f'{HDFS_BASE}/Sales/SalesOrderHeader'

# TODO: drop dulu kalau ada
spark_hive.sql('DROP TABLE IF EXISTS adventureworks.fact_sales_orders')  # TODO: 'DROP TABLE IF EXISTS adventureworks.fact_sales_orders'

# TODO: buat External Table dengan skema eksplisit + PARTITIONED BY
# Petunjuk: kolom partisi (order_year, order_month) JANGAN ditulis di list kolom utama
spark_hive.sql(f"""
    CREATE EXTERNAL TABLE adventureworks.fact_sales_orders (
        salesorderid           INT,
        revisionnumber         SMALLINT,
        orderdate              TIMESTAMP,
        duedate                TIMESTAMP,
        shipdate               TIMESTAMP,
        status                 SMALLINT,
        onlineorderflag        BOOLEAN,
        salesordernumber       STRING,
        purchaseordernumber    STRING,
        accountnumber          STRING,
        customerid             INT,
        salespersonid          INT,
        territoryid            INT,
        billtoaddressid        INT,
        shiptoaddressid        INT,
        shipmethodid           INT,
        creditcardid           INT,
        creditcardapprovalcode STRING,
        currencyrateid         INT,
        subtotal               DECIMAL(19,4),
        taxamt                 DECIMAL(19,4),
        freight                DECIMAL(19,4),
        totaldue               DECIMAL(19,4),
        comment                STRING,
        rowguid                STRING,
        modifieddate           TIMESTAMP
    )
    PARTITIONED BY (order_year INT, order_month INT)
    STORED AS PARQUET
    LOCATION '{orders_path}'
""")

# TODO: jalankan MSCK REPAIR TABLE agar partisi terdeteksi
spark_hive.sql('MSCK REPAIR TABLE adventureworks.fact_sales_orders')  # TODO

cnt = spark_hive.sql('SELECT COUNT(*) FROM adventureworks.fact_sales_orders').collect()[0][0]
print(f'  fact_sales_orders -> {cnt:,} rows')

spark_hive.sql('SHOW TABLES IN adventureworks').show(20)
spark_hive.stop()
print('Setup Hive External Tables selesai.')

  fact_sales_orders -> 31,465 rows
+--------------+-----------------+-----------+
|     namespace|        tableName|isTemporary|
+--------------+-----------------+-----------+
|adventureworks|     dim_customer|      false|
|adventureworks|       dim_person|      false|
|adventureworks|    dim_territory|      false|
|adventureworks|fact_sales_orders|      false|
+--------------+-----------------+-----------+

Setup Hive External Tables selesai.


## 0. Setup SparkSession

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('CaseC_CustomerRFM') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .config('spark.sql.parquet.enableVectorizedReader', 'True') \
    .enableHiveSupport() \
    .getOrCreate()

# Reference date — pakai tanggal max order sebagai "hari ini"
# (bukan tanggal sekarang, karena data AdventureWorks historis)
REFERENCE_DATE = '2014-07-31'  # tanggal terakhir di dataset AdventureWorks

print(f'Spark {spark.version} ready')
print(f'Reference date: {REFERENCE_DATE}')

Spark 3.5.0 ready
Reference date: 2014-07-31


## 1. Load Data dari Hive

In [8]:
# TODO: Load tabel yang dibutuhkan dari Hive
# Petunjuk: perlu fact_sales_orders (customerid, orderdate, totaldue)
#           dim_customer (customerid, personid, territoryid)
#           dim_person (businessentityid, firstname, lastname)
#           dim_territory (territoryid, name)

df_orders    = spark.table('adventureworks.fact_sales_orders')  # TODO
df_customer  = spark.table('adventureworks.dim_customer')  # TODO
df_person    = spark.table('adventureworks.dim_person')  # TODO
df_territory = spark.table('adventureworks.dim_territory')  # TODO

print('Row counts:')
for name, df in [('fact_sales_orders', df_orders), ('dim_customer', df_customer),
                  ('dim_person', df_person), ('dim_territory', df_territory)]:
    print(f'  {name}: {df.count():,}')

Row counts:
  fact_sales_orders: 31,465
  dim_customer: 19,820
  dim_person: 19,972
  dim_territory: 10


## 2. Eksplorasi Data

In [9]:
# TODO: Eksplorasi
# - Rentang tanggal order (min & max orderdate)
# - Jumlah customer unik
# - Distribusi total order per customer

print('=== Rentang Tanggal Order ===')
df_orders.select(F.min('orderdate').alias('min_date'), F.max('orderdate').alias('max_date')).show()
# TODO

print('\n=== Jumlah Customer Unik ===')
print(f"Total Customer Unik: {df_orders.select('customerid').distinct().count():,}")
# TODO

print('\n=== Distribusi Order Frequency per Customer ===')
df_orders.groupBy('customerid').count().describe().show()
# TODO: hitung frequency per customer, lalu describe()

=== Rentang Tanggal Order ===
+-------------------+-------------------+
|           min_date|           max_date|
+-------------------+-------------------+
|2011-05-31 00:00:00|2014-06-30 00:00:00|
+-------------------+-------------------+


=== Jumlah Customer Unik ===
Total Customer Unik: 19,119

=== Distribusi Order Frequency per Customer ===
+-------+-----------------+------------------+
|summary|       customerid|             count|
+-------+-----------------+------------------+
|  count|            19119|             19119|
|   mean|          20559.0|1.6457450703488676|
| stddev|5519.324233998197|1.4570539244635008|
|    min|            11000|                 1|
|    max|            30118|                28|
+-------+-----------------+------------------+



## 3. Hitung R, F, M per Customer

**Tugas:** Dari `fact_sales_orders`, hitung Recency, Frequency, Monetary per customer.

In [10]:
# TODO: Hitung RFM raw values
# Recency   = datediff(reference_date, max(orderdate))  ← hari sejak order terakhir
# Frequency = count(salesorderid)                       ← total order
# Monetary  = round(sum(totaldue), 2)                   ← total spending

df_rfm_raw = df_orders \
    .groupBy('customerid') \
    .agg(
        F.datediff(F.lit(REFERENCE_DATE), F.max('orderdate')).alias('recency_days'),  # TODO: recency_days
        F.count('salesorderid').alias('frequency'),  # TODO: frequency
        F.round(F.sum('totaldue'), 2).alias('monetary'),  # TODO: monetary
        F.max('orderdate').alias('last_order_date')
    )

print(f'Total customers: {df_rfm_raw.count():,}')
df_rfm_raw.describe().show()
df_rfm_raw.orderBy(F.asc('recency_days')).show(10)

Total customers: 19,119
+-------+-----------------+------------------+------------------+------------------+
|summary|       customerid|      recency_days|         frequency|          monetary|
+-------+-----------------+------------------+------------------+------------------+
|  count|            19119|             19119|             19119|             19119|
|   mean|          20559.0|221.26748260892307|1.6457450703488676|       6444.729527|
| stddev|5519.324233998197|150.42360489918482|1.4570539244635008|43756.276043011814|
|    min|            11000|                31|                 1|              1.52|
|    max|            30118|              1157|                28|         989184.08|
+-------+-----------------+------------------+------------------+------------------+

+----------+------------+---------+--------+-------------------+
|customerid|recency_days|frequency|monetary|    last_order_date|
+----------+------------+---------+--------+-------------------+
|     16402|   

## 4. Scoring R, F, M (Quartile-based)

Gunakan `ntile(4)` window function untuk membagi customer menjadi 4 kuartil.

> **Ingat:** R-score **terbalik** — recency kecil = baru beli = score 4 (bagus).

In [11]:
# TODO: Hitung R, F, M score menggunakan ntile(4)
# Petunjuk:
# - R score: ntile(4) OVER (ORDER BY recency_days DESC)  ← terbalik! desc = kecil dapat rank 1→4
#   tapi kita mau recency kecil = score 4, jadi pakai:
#   ntile(4) OVER (ORDER BY recency_days ASC) lalu (5 - ntile_value)
#   ATAU langsung: ntile(4) OVER (ORDER BY recency_days DESC)
# - F score: ntile(4) OVER (ORDER BY frequency ASC)
# - M score: ntile(4) OVER (ORDER BY monetary ASC)

# HARI TERBESAR (Lama tidak beli) = NTILE 1 (Skor 1)
win_r = Window.orderBy(F.desc('recency_days'))   
# FREKUENSI TERKECIL = NTILE 1 (Skor 1)
win_f = Window.orderBy(F.asc('frequency'))        
# BELANJA TERKECIL = NTILE 1 (Skor 1)
win_m = Window.orderBy(F.asc('monetary'))         

df_scored = df_rfm_raw \
    .withColumn('r_score', F.ntile(4).over(win_r)) \
    .withColumn('f_score', F.ntile(4).over(win_f)) \
    .withColumn('m_score', F.ntile(4).over(win_m))

df_scored = df_scored.withColumn(
    'rfm_score',
    F.concat(F.col('r_score').cast('string'),
             F.col('f_score').cast('string'),
             F.col('m_score').cast('string'))
)

print('=== Score Distribution ===')
df_scored.groupBy('r_score', 'f_score', 'm_score').count().orderBy('r_score','f_score','m_score').show(20)

=== Score Distribution ===
+-------+-------+-------+-----+
|r_score|f_score|m_score|count|
+-------+-------+-------+-----+
|      1|      1|      1| 1268|
|      1|      1|      2|  917|
|      1|      1|      3| 1098|
|      1|      1|      4|  256|
|      1|      3|      1|   20|
|      1|      3|      2|   90|
|      1|      3|      3|  223|
|      1|      3|      4|  733|
|      1|      4|      2|    8|
|      1|      4|      3|    9|
|      1|      4|      4|  158|
|      2|      1|      1|  479|
|      2|      1|      2|  351|
|      2|      1|      3|  411|
|      2|      2|      1|  744|
|      2|      2|      2|  489|
|      2|      2|      3|  482|
|      2|      3|      1|   51|
|      2|      3|      2|  266|
|      2|      3|      3|  297|
+-------+-------+-------+-----+
only showing top 20 rows



## 5. Segmentasi Customer

In [12]:
# TODO: Assign segment berdasarkan R, F, M score
# Gunakan F.when().when().otherwise()
#
# Champion      : R>=3 AND F>=3 AND M>=3
# Loyal         : F>=3 AND M>=3 (bukan Champion)
# Potential Loyal: R>=3 AND F>=2
# At Risk       : R<=2 AND F>=3 AND M>=3
# Lost          : R=1 AND F<=2
# New Customer  : R=4 AND F=1
# Others        : otherwise

r = F.col('r_score')
f = F.col('f_score')
m = F.col('m_score')

df_segmented = df_scored.withColumn(
    'rfm_segment',
    F.when((r >= 3) & (f >= 3) & (m >= 3), 'Champion')
     .when((f >= 3) & (m >= 3), 'Loyal')  # TODO: Loyal
     .when((r >= 3) & (f >= 2), 'Potential Loyal')  # TODO: Potential Loyal
     .when((r <= 2) & (f >= 3) & (m >= 3), 'At Risk')  # TODO: At Risk
     .when((r == 1) & (f <= 2), 'Lost')  # TODO: Lost
     .when((r == 4) & (f == 1), 'New Customer')  # TODO: New Customer
     .otherwise('Others')
)

print('=== Distribusi Segmen ===')
df_segmented.groupBy('rfm_segment') \
    .agg(
        F.count('customerid').alias('jumlah_customer'),
        F.round(F.avg('monetary'), 2).alias('avg_spending'),
        F.round(F.avg('frequency'), 1).alias('avg_frequency')
    ) \
    .orderBy(F.desc('jumlah_customer')) \
    .show()

=== Distribusi Segmen ===
+---------------+---------------+------------+-------------+
|    rfm_segment|jumlah_customer|avg_spending|avg_frequency|
+---------------+---------------+------------+-------------+
|Potential Loyal|           5840|      342.06|          1.3|
|       Champion|           3719|    24099.62|          2.8|
|           Lost|           3539|      873.29|          1.0|
|         Others|           3442|      443.52|          1.2|
|          Loyal|           2579|    10459.68|          2.3|
+---------------+---------------+------------+-------------+



## 6. Enrich dengan Data Customer

In [13]:
# TODO: Join dengan dim_customer, dim_person, dim_territory
# Hasilnya: satu baris per customer dengan nama, wilayah, dan semua RFM info

df_final_rfm = df_segmented \
    .join(df_customer.select('customerid', 'personid', 'territoryid'), 'customerid', 'left') \
    .join(
        df_person.select(
            F.col('businessentityid').alias('personid'),
            F.concat_ws(' ', 'firstname', 'lastname').alias('customer_name')
        ), 'personid', 'left') \
    .join(F.broadcast(
        df_territory.select(
            'territoryid',
            F.col('name').alias('territory_name')
        )), 'territoryid', 'left') \
    .select(
        'customerid', 'customer_name', 'territory_name',
        'last_order_date', 'recency_days', 'frequency', 'monetary',
        'r_score', 'f_score', 'm_score', 'rfm_score', 'rfm_segment'
    ) \
    .withColumn('load_timestamp', F.current_timestamp())

print(f'Final RFM rows: {df_final_rfm.count():,}')
df_final_rfm.show(10, truncate=False)

Final RFM rows: 19,119
+----------+----------------+--------------+-------------------+------------+---------+--------+-------+-------+-------+---------+---------------+--------------------------+
|customerid|customer_name   |territory_name|last_order_date    |recency_days|frequency|monetary|r_score|f_score|m_score|rfm_score|rfm_segment    |load_timestamp            |
+----------+----------------+--------------+-------------------+------------+---------+--------+-------+-------+-------+---------+---------------+--------------------------+
|30078     |Tony Ting       |Northeast     |2013-06-30 00:00:00|396         |1        |1.52    |1      |1      |1      |111      |Lost           |2026-05-15 02:38:32.921703|
|28094     |Melanie Peterson|Northwest     |2013-07-24 00:00:00|372         |1        |2.53    |1      |1      |1      |111      |Lost           |2026-05-15 02:38:32.921703|
|28300     |Abigail Bennett |Northwest     |2013-09-11 00:00:00|323         |1        |2.53    |1      |1  

## 7. Load — Simpan ke Hive

In [14]:
spark.sql('CREATE DATABASE IF NOT EXISTS adventureworks_curated')

# TODO: simpan ke adventureworks_curated.fact_customer_rfm
# mode: overwrite, saveAsTable

df_final_rfm.write \
    .mode('overwrite') \
    .saveAsTable('adventureworks_curated.fact_customer_rfm')

print('=== Verifikasi ===')
df_verify = spark.table('adventureworks_curated.fact_customer_rfm')
print(f'Total rows: {df_verify.count():,}')
df_verify.show(5)

=== Verifikasi ===
Total rows: 19,119
+----------+----------------+--------------+-------------------+------------+---------+--------+-------+-------+-------+---------+-----------+--------------------+
|customerid|   customer_name|territory_name|    last_order_date|recency_days|frequency|monetary|r_score|f_score|m_score|rfm_score|rfm_segment|      load_timestamp|
+----------+----------------+--------------+-------------------+------------+---------+--------+-------+-------+-------+---------+-----------+--------------------+
|     30078|       Tony Ting|     Northeast|2013-06-30 00:00:00|         396|        1|    1.52|      1|      1|      1|      111|       Lost|2026-05-15 02:38:...|
|     28094|Melanie Peterson|     Northwest|2013-07-24 00:00:00|         372|        1|    2.53|      1|      1|      1|      111|       Lost|2026-05-15 02:38:...|
|     28300| Abigail Bennett|     Northwest|2013-09-11 00:00:00|         323|        1|    2.53|      1|      1|      1|      111|       Lost|

## 8. Analytic Queries

**Wajib: minimal 3 query yang menjawab business question**

In [16]:
# ── Query 1 (WAJIB): Top 20 Champion Customers ─────────────────────────────
# Tampilkan customer segment Champion, sort by monetary desc
# Kolom: customer_name, territory_name, recency_days, frequency, monetary, rfm_score

print('=== Query 1: Top 20 Champion Customers ===')
spark.sql("""
    SELECT 
        customer_name,
        territory_name,
        recency_days,
        frequency,
        monetary,
        rfm_score
    FROM adventureworks_curated.fact_customer_rfm
    WHERE rfm_segment = 'Champion'
    ORDER BY monetary DESC
    LIMIT 20
""").show(20, truncate=False)


=== Query 1: Top 20 Champion Customers ===
+------------------+--------------+------------+---------+---------+---------+
|customer_name     |territory_name|recency_days|frequency|monetary |rfm_score|
+------------------+--------------+------------+---------+---------+---------+
|Roger Harui       |Northwest     |122         |12       |989184.08|344      |
|Andrew Dixon      |Southeast     |91          |12       |961675.86|444      |
|Reuben D'sa       |Canada        |91          |12       |954021.92|444      |
|Robert Vessa      |Southwest     |122         |12       |919801.82|344      |
|Ryan Calafato     |Canada        |152         |12       |901346.86|344      |
|Joseph Castellucio|Canada        |91          |12       |887090.41|444      |
|Kirk DeGrasse     |Northeast     |91          |8        |841866.55|444      |
|Lindsey Camacho   |Northwest     |122         |12       |834475.93|344      |
|Robin McGuigan    |Canada        |152         |12       |824331.77|344      |
|Stacey C

In [17]:
# ── Query 2 (WAJIB): Distribusi Segmen + Revenue Kontribusi ────────────────
# Tampilkan: rfm_segment, jumlah_customer, pct_customer, total_revenue, pct_revenue
# pct = % dari total

print('=== Query 2: Distribusi Segmen & Revenue Contribution ===')
spark.sql("""
    SELECT
        rfm_segment,
        COUNT(customerid)                                           AS jumlah_customer,
        ROUND(COUNT(customerid) * 100.0 / SUM(COUNT(customerid)) OVER (), 2) AS pct_customer,
        ROUND(SUM(monetary), 2)                                     AS total_revenue,
        ROUND(SUM(monetary) * 100.0 / SUM(SUM(monetary)) OVER (), 2) AS pct_revenue
    FROM adventureworks_curated.fact_customer_rfm
    GROUP BY rfm_segment
    ORDER BY total_revenue DESC
""").show(truncate=False)
# TODO: Gunakan window function sum() OVER () untuk hitung total keseluruhan

=== Query 2: Distribusi Segmen & Revenue Contribution ===
+---------------+---------------+------------+-------------+-----------+
|rfm_segment    |jumlah_customer|pct_customer|total_revenue|pct_revenue|
+---------------+---------------+------------+-------------+-----------+
|Champion       |3719           |19.45       |89626497.25  |72.74      |
|Loyal          |2579           |13.49       |26975502.50  |21.89      |
|Lost           |3539           |18.51       |3090558.81   |2.51       |
|Potential Loyal|5840           |30.55       |1997643.37   |1.62       |
|Others         |3442           |18.00       |1526581.90   |1.24       |
+---------------+---------------+------------+-------------+-----------+



In [18]:
# ── Query 3 (WAJIB): At Risk & Lost Customers per Territory ────────────────
# Tampilkan pelanggan yang perlu di-winback
# Kolom: territory_name, rfm_segment, count, avg_monetary, avg_recency_days

print('=== Query 3: At Risk & Lost Customers per Territory ===')
# TODO
spark.sql("""
    SELECT
        territory_name,
        rfm_segment,
        COUNT(customerid)            AS count,
        ROUND(AVG(monetary), 2)      AS avg_monetary,
        ROUND(AVG(recency_days), 0)  AS avg_recency_days
    FROM adventureworks_curated.fact_customer_rfm
    WHERE rfm_segment IN ('At Risk', 'Lost')
    GROUP BY territory_name, rfm_segment
    ORDER BY territory_name, rfm_segment
""").show(50, truncate=False)

=== Query 3: At Risk & Lost Customers per Territory ===
+--------------+-----------+-----+------------+----------------+
|territory_name|rfm_segment|count|avg_monetary|avg_recency_days|
+--------------+-----------+-----+------------+----------------+
|Australia     |Lost       |299  |602.26      |345.0           |
|Canada        |Lost       |320  |1389.26     |529.0           |
|Central       |Lost       |4    |877.33      |590.0           |
|France        |Lost       |386  |628.35      |399.0           |
|Germany       |Lost       |364  |582.13      |370.0           |
|Northeast     |Lost       |4    |584.51      |583.0           |
|Northwest     |Lost       |788  |872.72      |429.0           |
|Southeast     |Lost       |8    |1236.53     |510.0           |
|Southwest     |Lost       |1075 |1039.26     |430.0           |
|United Kingdom|Lost       |291  |655.80      |353.0           |
+--------------+-----------+-----+------------+----------------+



In [19]:
# ── Query 4 (BONUS): Average RFM metrics per Segment ──────────────────────
# Tampilkan profil rata-rata tiap segmen
# Kolom: rfm_segment, avg_recency, avg_frequency, avg_monetary, count_customers
# Sort: avg_monetary desc

print('=== Query 4 (Bonus): Profil Rata-rata per Segmen ===')
# TODO
spark.sql("""
    SELECT
        rfm_segment,
        ROUND(AVG(recency_days), 1)  AS avg_recency,
        ROUND(AVG(frequency), 1)     AS avg_frequency,
        ROUND(AVG(monetary), 2)      AS avg_monetary,
        COUNT(customerid)            AS count_customers
    FROM adventureworks_curated.fact_customer_rfm
    GROUP BY rfm_segment
    ORDER BY avg_monetary DESC
""").show(truncate=False)

=== Query 4 (Bonus): Profil Rata-rata per Segmen ===
+---------------+-----------+-------------+------------+---------------+
|rfm_segment    |avg_recency|avg_frequency|avg_monetary|count_customers|
+---------------+-----------+-------------+------------+---------------+
|Champion       |115.2      |2.8          |24099.62    |3719           |
|Loyal          |305.6      |2.3          |10459.68    |2579           |
|Lost           |416.1      |1.0          |873.29      |3539           |
|Others         |248.1      |1.2          |443.52      |3442           |
|Potential Loyal|117.7      |1.3          |342.06      |5840           |
+---------------+-----------+-------------+------------+---------------+



In [20]:
# ── Query 5 (BONUS): Segmen Paling Banyak per Territory ───────────────────
# Untuk tiap territory, segmen mana yang paling dominan?
# Gunakan window function rank() partitionBy territory_name, orderBy desc count

print('=== Query 5 (Bonus): Dominant Segment per Territory ===')
# TODO
spark.sql("""
    SELECT territory_name, rfm_segment, jumlah
    FROM (
        SELECT
            territory_name,
            rfm_segment,
            COUNT(customerid) AS jumlah,
            RANK() OVER (PARTITION BY territory_name ORDER BY COUNT(customerid) DESC) AS rnk
        FROM adventureworks_curated.fact_customer_rfm
        GROUP BY territory_name, rfm_segment
    )
    WHERE rnk = 1
    ORDER BY jumlah DESC
""").show(truncate=False)

=== Query 5 (Bonus): Dominant Segment per Territory ===
+--------------+---------------+------+
|territory_name|rfm_segment    |jumlah|
+--------------+---------------+------+
|Southwest     |Potential Loyal|1432  |
|Northwest     |Potential Loyal|1188  |
|Australia     |Champion       |1077  |
|Canada        |Potential Loyal|621   |
|United Kingdom|Potential Loyal|619   |
|France        |Potential Loyal|557   |
|Germany       |Potential Loyal|523   |
|Southeast     |Champion       |48    |
|Northeast     |Champion       |37    |
|Central       |Champion       |36    |
+--------------+---------------+------+



In [21]:
spark.stop()
print('Done!')

Done!
